# Pricing Walkthrough: PR 3a (`marrid_3_swaptions` — Linear Multi-Curve Extensions)

Interactive walkthrough for IMM-date 3M FRAs, negative-rate OIS (`EUR_OIS`), and EUR/USD cross-currency basis swaps with initial/final notional exchanges over the `PR_CASHFLOW` table.

In [1]:
import datetime as dt
from pathlib import Path

from pricing.rack import (
    FraSpec,
    SwapSpec,
    XccyBasisSwapSpec,
    expand_fra_to_cashflows,
    expand_swap_to_cashflows,
    expand_xccy_basis_swap_to_cashflows,
    fit_curve_from_cashflows,
    fit_zero_curve,
    next_imm_dates,
    price_cashflows_python,
    run_pure_pricing_duckdb,
)

PRICING_DIR = Path.cwd().resolve()
while not (PRICING_DIR / "projects" / "pricing" / "model.pure").is_file():
    PRICING_DIR = PRICING_DIR.parent
PRICING_DIR = PRICING_DIR / "projects" / "pricing"
VAL_DATE = dt.date(2026, 1, 2)
HOLIDAYS = {dt.date(2026, 1, 1), dt.date(2026, 12, 25)}

usd_bms = [
    SwapSpec("BM_6M",  "OIS", "USD", "USD_OIS", "USD_OIS", VAL_DATE,   6, 6, 1_000_000.0, 0.0410),
    SwapSpec("BM_1Y",  "OIS", "USD", "USD_OIS", "USD_OIS", VAL_DATE,  12, 6, 1_000_000.0, 0.0420),
    SwapSpec("BM_2Y",  "OIS", "USD", "USD_OIS", "USD_OIS", VAL_DATE,  24, 6, 1_000_000.0, 0.0435),
    SwapSpec("BM_5Y",  "OIS", "USD", "USD_OIS", "USD_OIS", VAL_DATE,  60, 6, 1_000_000.0, 0.0470),
    SwapSpec("BM_10Y", "OIS", "USD", "USD_OIS", "USD_OIS", VAL_DATE, 120, 6, 1_000_000.0, 0.0500),
]
usd_ois = fit_zero_curve(usd_bms, VAL_DATE, HOLIDAYS, "USD_OIS", "LOG_LINEAR_DF")
print("Using projects/pricing:", PRICING_DIR, "| USD_OIS knots:", len(usd_ois))

Using projects/pricing: /home/dunst/wk/legend-lite/projects/pricing | USD_OIS knots: 5


## Step 1: Hybrid IMM FRA + Swap Multi-Curve Fit (`USD_3M` over `USD_OIS`)

In [2]:
imms = next_imm_dates(VAL_DATE, 4)
print("Next 4 IMM dates:", imms)
usd_3m_bms = [
    expand_fra_to_cashflows(FraSpec("IMM_1", "USD", "USD_OIS", "USD_3M", imms[0], imms[1], 5_000_000.0, 0.0430), VAL_DATE, HOLIDAYS),
    expand_fra_to_cashflows(FraSpec("IMM_2", "USD", "USD_OIS", "USD_3M", imms[1], imms[2], 5_000_000.0, 0.0440), VAL_DATE, HOLIDAYS),
    expand_fra_to_cashflows(FraSpec("IMM_3", "USD", "USD_OIS", "USD_3M", imms[2], imms[3], 5_000_000.0, 0.0450), VAL_DATE, HOLIDAYS),
    expand_swap_to_cashflows(SwapSpec("IRS_2Y", "IRS", "USD", "USD_OIS", "USD_3M", VAL_DATE, 24, 6, 5_000_000.0, 0.0465), VAL_DATE, HOLIDAYS),
    expand_swap_to_cashflows(SwapSpec("IRS_5Y", "IRS", "USD", "USD_OIS", "USD_3M", VAL_DATE, 60, 6, 5_000_000.0, 0.0495), VAL_DATE, HOLIDAYS),
]
usd_3m = fit_curve_from_cashflows(usd_3m_bms, "USD_3M", "LOG_LINEAR_DF", fixed_knots=usd_ois, initial_rate=0.045)
bm_npvs = run_pure_pricing_duckdb(usd_ois + usd_3m, [cf for b in usd_3m_bms for cf in b], projects_dir=PRICING_DIR)
print("USD_3M benchmark Pure NPVs:", {k[1]: round(v, 8) for k, v in bm_npvs.items()})

Next 4 IMM dates: [datetime.date(2026, 3, 18), datetime.date(2026, 6, 17), datetime.date(2026, 9, 16), datetime.date(2026, 12, 16)]


USD_3M benchmark Pure NPVs: {'IMM_1': -0.0, 'IMM_2': -0.0, 'IMM_3': 0.0, 'IRS_2Y': -0.0, 'IRS_5Y': -0.0}


## Step 2: Negative-Rate `EUR_OIS` & EUR/USD XCCY Basis Swap (`EUR_DISC_USD`)

In [3]:
eur_bms = [
    SwapSpec("EUR_6M",  "OIS", "EUR", "EUR_OIS", "EUR_OIS", VAL_DATE,   6, 6, 1_000_000.0, -0.015),
    SwapSpec("EUR_2Y",  "OIS", "EUR", "EUR_OIS", "EUR_OIS", VAL_DATE,  24, 6, 1_000_000.0, -0.008),
    SwapSpec("EUR_5Y",  "OIS", "EUR", "EUR_OIS", "EUR_OIS", VAL_DATE,  60, 6, 1_000_000.0,  0.002),
    SwapSpec("EUR_10Y", "OIS", "EUR", "EUR_OIS", "EUR_OIS", VAL_DATE, 120, 6, 1_000_000.0,  0.010),
]
eur_ois = fit_zero_curve(eur_bms, VAL_DATE, HOLIDAYS, "EUR_OIS", "LOG_LINEAR_DF")
spot = 1.10
xccy_bms = [
    expand_xccy_basis_swap_to_cashflows(
        XccyBasisSwapSpec(f"XCCY_{m}M", "EUR", "USD", "EUR_DISC_USD", "EUR_OIS", "USD_OIS", "USD_OIS", VAL_DATE, m, 6, 1_000_000.0, spot, sp),
        VAL_DATE, HOLIDAYS,
    )
    for m, sp in [(12, -0.0012), (24, -0.0015), (60, -0.0018), (120, -0.0020)]
]
eur_disc_usd = fit_curve_from_cashflows(xccy_bms, "EUR_DISC_USD", "LOG_LINEAR_DF", fixed_knots=usd_ois + eur_ois, initial_rate=-0.005)
all_knots = usd_ois + eur_ois + eur_disc_usd

xccy_spec = XccyBasisSwapSpec("PF_XCCY_5Y", "EUR", "USD", "EUR_DISC_USD", "EUR_OIS", "USD_OIS", "USD_OIS", dt.date(2026, 4, 2), 60, 6, 2_000_000.0, spot, -0.0010)
rep_cfs = expand_xccy_basis_swap_to_cashflows(xccy_spec, VAL_DATE, HOLIDAYS, in_reporting_ccy=True)
nat_cfs = expand_xccy_basis_swap_to_cashflows(xccy_spec, VAL_DATE, HOLIDAYS, in_reporting_ccy=False)
py_x = price_cashflows_python(rep_cfs, all_knots)[("BASE", "PF_XCCY_5Y", "USD", "USD")]
sql_x = run_pure_pricing_duckdb(all_knots, rep_cfs, projects_dir=PRICING_DIR)[("BASE", "PF_XCCY_5Y", "USD", "USD")]
nat_npvs = run_pure_pricing_duckdb(all_knots, nat_cfs, projects_dir=PRICING_DIR)
print(f"PF_XCCY_5Y (USD, CSA=USD): py={py_x:,.2f}  duckdb={sql_x:,.2f}  diff={abs(py_x - sql_x):.2e}")
print("Native leg PVs by (ccy, csa):", {(k[2], k[3]): round(v, 2) for k, v in nat_npvs.items()})

PF_XCCY_5Y (USD, CSA=USD): py=-12,123.58  duckdb=-12,123.58  diff=2.33e-10
Native leg PVs by (ccy, csa): {('EUR', 'USD'): -11021.43, ('USD', 'USD'): -0.0}
